# 20. Serialization, schemas, enums, and snapshots

**Level:** Intermediate to advanced

**Learning outcome:** Export deliberate public fields, reject unsupported schema versions, and control snapshot ownership.

**Prerequisites:** 03 and 11: validation, records, and copying.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](19_Testing_Fakes_Mocks_and_Invariants.ipynb) | [Next notebook](21_Descriptors_and_Slots.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Concept notes and mental model

**Level:** Intermediate to advanced

**Learning goal:** Serialize domain values deliberately and reject unsupported or malformed records.

**Serialization** converts a model to a transportable representation, not to executable code.
JSON supports a limited set of primitive shapes. An ipaddress value or Enum needs an explicit
representation. Avoid dumping arbitrary __dict__ contents: internal fields, caches, and
credentials can leak, and the resulting schema changes when implementation details change.

**Deserialization is validation.** Require the fields and types your schema promises, reject
unknown schema versions, and normalize only intentionally. JSON decoding alone says nothing
about whether a VLAN, hostname, or MTU is acceptable. bool is a subclass of int in Python, so
strict numeric schemas may need a deliberate check.

**Version your records.** A schema_version field creates an explicit compatibility boundary.
Migrate old formats intentionally; do not silently treat an unsupported version as today's
format. Decide whether unknown fields are rejected, ignored, or preserved. This chapter's
small schemas prefer explicit rejection for clarity.

**Enums** give named states with a defined allowed set. Store the stable value rather than
relying on a debug repr. A redacted report should be constructed from an allowlist of public
fields. Simply deleting a key called password does not guarantee all sensitive data is gone.

**Snapshots** should express ownership. Returning a copied mapping prevents some accidental
mutation; immutable nested values or deeper copies may be needed for nested structures.
These exercises use strings, integers, tuples, and JSON records, never real secrets.

**Pause and explain:** Explain why JSON decoding is not schema validation and why dumping __dict__ is risky.

## Worked-example structure

This notebook contains **5 worked patterns and 50 executable scenario examples**. Each pattern explains its purpose, logic, common mistakes, and extension exercise. Its ten scenarios vary the input data; they are related practice, not ten unrelated algorithms. Original Example IDs are retained so you can locate the same exercise in `AI_oops_learning.ipynb`.

The input table lists only fields needed by that pattern. Expected output is shown before each code cell and a saved executed result appears afterward. Expected rejection branches are checked explicitly; unexpected exceptions are not silently ignored.

## Pattern 071: JSON round trip through explicit fields

**Core concept:** A domain model controls its public serialized schema.

**Network-engineering purpose:** An inventory record can be exported without exposing unrelated implementation attributes.

### Read the implementation in this order
1. Create a dataclass.
2. Serialize an explicit field mapping using json.dumps.
3. Decode and reconstruct through declared keys.
4. Check value equality after round trip.

**Common mistake / boundary:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

**Practice after the ten worked scenarios:** Add a from_json boundary that rejects unknown fields and wrong types.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0701 - JSON round trip through explicit fields: branch scenario

**Scenario 01/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'branch'` |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "branch-edge-01", "site": "branch"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [1]:
# Example 0701: JSON round trip through explicit fields - branch
case = {'site': 'branch', 'hostname': 'branch-edge-01'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "branch-edge-01", "site": "branch"}


### Example 0702 - JSON round trip through explicit fields: campus scenario

**Scenario 02/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'campus'` |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "campus-access-02", "site": "campus"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [2]:
# Example 0702: JSON round trip through explicit fields - campus
case = {'site': 'campus', 'hostname': 'campus-access-02'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "campus-access-02", "site": "campus"}


### Example 0703 - JSON round trip through explicit fields: dc scenario

**Scenario 03/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'dc'` |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "dc-leaf-03", "site": "dc"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [3]:
# Example 0703: JSON round trip through explicit fields - dc
case = {'site': 'dc', 'hostname': 'dc-leaf-03'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "dc-leaf-03", "site": "dc"}


### Example 0704 - JSON round trip through explicit fields: wan scenario

**Scenario 04/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'wan'` |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "wan-border-04", "site": "wan"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [4]:
# Example 0704: JSON round trip through explicit fields - wan
case = {'site': 'wan', 'hostname': 'wan-border-04'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "wan-border-04", "site": "wan"}


### Example 0705 - JSON round trip through explicit fields: lab scenario

**Scenario 05/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'lab'` |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "lab-router-05", "site": "lab"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [5]:
# Example 0705: JSON round trip through explicit fields - lab
case = {'site': 'lab', 'hostname': 'lab-router-05'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "lab-router-05", "site": "lab"}


### Example 0706 - JSON round trip through explicit fields: wireless scenario

**Scenario 06/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'wireless'` |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "wireless-dist-06", "site": "wireless"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [6]:
# Example 0706: JSON round trip through explicit fields - wireless
case = {'site': 'wireless', 'hostname': 'wireless-dist-06'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "wireless-dist-06", "site": "wireless"}


### Example 0707 - JSON round trip through explicit fields: remote scenario

**Scenario 07/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'remote'` |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "remote-gateway-07", "site": "remote"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [7]:
# Example 0707: JSON round trip through explicit fields - remote
case = {'site': 'remote', 'hostname': 'remote-gateway-07'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "remote-gateway-07", "site": "remote"}


### Example 0708 - JSON round trip through explicit fields: core scenario

**Scenario 08/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'core'` |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "core-router-08", "site": "core"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [8]:
# Example 0708: JSON round trip through explicit fields - core
case = {'site': 'core', 'hostname': 'core-router-08'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "core-router-08", "site": "core"}


### Example 0709 - JSON round trip through explicit fields: mgmt scenario

**Scenario 09/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'mgmt'` |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "mgmt-switch-09", "site": "mgmt"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [9]:
# Example 0709: JSON round trip through explicit fields - mgmt
case = {'site': 'mgmt', 'hostname': 'mgmt-switch-09'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "mgmt-switch-09", "site": "mgmt"}


### Example 0710 - JSON round trip through explicit fields: backup scenario

**Scenario 10/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'backup'` |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "backup-edge-10", "site": "backup"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [10]:
# Example 0710: JSON round trip through explicit fields - backup
case = {'site': 'backup', 'hostname': 'backup-edge-10'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "backup-edge-10", "site": "backup"}


## Pattern 072: Versioned schema rejects unsupported records

**Core concept:** A version field prevents accidental interpretation of incompatible data.

**Network-engineering purpose:** A saved intent should not be applied using assumptions from a different record format.

### Read the implementation in this order
1. Require exactly the declared keys.
2. Check version and strict MTU type.
3. Validate the model's MTU range.
4. Accept a valid record and reject a newer version.

**Common mistake / boundary:** Do not silently default an unknown version to a known schema.

**Practice after the ten worked scenarios:** Implement an explicit version-0-to-version-1 migration as a separate function.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0711 - Versioned schema rejects unsupported records: branch scenario

**Scenario 01/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [11]:
# Example 0711: Versioned schema rejects unsupported records - branch
case = {'mtu': 1500}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0712 - Versioned schema rejects unsupported records: campus scenario

**Scenario 02/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [12]:
# Example 0712: Versioned schema rejects unsupported records - campus
case = {'mtu': 9000}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0713 - Versioned schema rejects unsupported records: dc scenario

**Scenario 03/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [13]:
# Example 0713: Versioned schema rejects unsupported records - dc
case = {'mtu': 9216}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0714 - Versioned schema rejects unsupported records: wan scenario

**Scenario 04/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [14]:
# Example 0714: Versioned schema rejects unsupported records - wan
case = {'mtu': 576}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0715 - Versioned schema rejects unsupported records: lab scenario

**Scenario 05/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [15]:
# Example 0715: Versioned schema rejects unsupported records - lab
case = {'mtu': 1600}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0716 - Versioned schema rejects unsupported records: wireless scenario

**Scenario 06/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [16]:
# Example 0716: Versioned schema rejects unsupported records - wireless
case = {'mtu': 1500}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0717 - Versioned schema rejects unsupported records: remote scenario

**Scenario 07/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [17]:
# Example 0717: Versioned schema rejects unsupported records - remote
case = {'mtu': 2000}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0718 - Versioned schema rejects unsupported records: core scenario

**Scenario 08/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [18]:
# Example 0718: Versioned schema rejects unsupported records - core
case = {'mtu': 4470}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0719 - Versioned schema rejects unsupported records: mgmt scenario

**Scenario 09/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [19]:
# Example 0719: Versioned schema rejects unsupported records - mgmt
case = {'mtu': 9100}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0720 - Versioned schema rejects unsupported records: backup scenario

**Scenario 10/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [20]:
# Example 0720: Versioned schema rejects unsupported records - backup
case = {'mtu': 8192}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


## Pattern 073: Enum represents an allowed state set

**Core concept:** Named states prevent arbitrary strings from spreading through the model.

**Network-engineering purpose:** An interface observation can represent up and down with a defined vocabulary.

### Read the implementation in this order
1. Declare an Enum with stable string values.
2. Parse an external token using the enum constructor.
3. Use the enum member in the object.
4. Reject an unrecognized token.

**Common mistake / boundary:** A real observation model may need unknown or unavailable distinct from down.

**Practice after the ten worked scenarios:** Add an UNKNOWN member and define exactly which inputs may produce it.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0721 - Enum represents an allowed state set: branch scenario

**Scenario 01/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `True` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
UP up
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [21]:
# Example 0721: Enum represents an allowed state set - branch
case = {'healthy': True}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected
UP up


### Example 0722 - Enum represents an allowed state set: campus scenario

**Scenario 02/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `False` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
DOWN down
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [22]:
# Example 0722: Enum represents an allowed state set - campus
case = {'healthy': False}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected
DOWN down


### Example 0723 - Enum represents an allowed state set: dc scenario

**Scenario 03/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `True` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
UP up
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [23]:
# Example 0723: Enum represents an allowed state set - dc
case = {'healthy': True}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected
UP up


### Example 0724 - Enum represents an allowed state set: wan scenario

**Scenario 04/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `True` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
UP up
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [24]:
# Example 0724: Enum represents an allowed state set - wan
case = {'healthy': True}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected
UP up


### Example 0725 - Enum represents an allowed state set: lab scenario

**Scenario 05/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `False` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
DOWN down
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [25]:
# Example 0725: Enum represents an allowed state set - lab
case = {'healthy': False}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected


DOWN down


### Example 0726 - Enum represents an allowed state set: wireless scenario

**Scenario 06/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `True` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
UP up
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [26]:
# Example 0726: Enum represents an allowed state set - wireless
case = {'healthy': True}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected


UP up


### Example 0727 - Enum represents an allowed state set: remote scenario

**Scenario 07/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `False` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
DOWN down
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [27]:
# Example 0727: Enum represents an allowed state set - remote
case = {'healthy': False}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected


DOWN down


### Example 0728 - Enum represents an allowed state set: core scenario

**Scenario 08/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `True` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
UP up
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [28]:
# Example 0728: Enum represents an allowed state set - core
case = {'healthy': True}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected


UP up


### Example 0729 - Enum represents an allowed state set: mgmt scenario

**Scenario 09/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `False` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
DOWN down
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [29]:
# Example 0729: Enum represents an allowed state set - mgmt
case = {'healthy': False}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected


DOWN down


### Example 0730 - Enum represents an allowed state set: backup scenario

**Scenario 10/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `True` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
UP up
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [30]:
# Example 0730: Enum represents an allowed state set - backup
case = {'healthy': True}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected
UP up


## Pattern 074: Allowlist a public report

**Core concept:** Build the safe report from approved fields rather than copying every internal field.

**Network-engineering purpose:** An inventory object may contain internal notes that do not belong in a user-facing summary.

### Read the implementation in this order
1. Keep internal metadata separate.
2. Return only the approved hostname and site.
3. Verify the exact public key set.
4. Check internal-only data is not in the serialized report.

**Common mistake / boundary:** An allowlist limits accidental disclosure but does not replace a data-handling review.

**Practice after the ten worked scenarios:** Add a nested field and decide explicitly whether it is safe for the public report.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0731 - Allowlist a public report: branch scenario

**Scenario 01/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'branch'` |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "branch-edge-01", "site": "branch"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [31]:
# Example 0731: Allowlist a public report - branch
case = {'site': 'branch', 'hostname': 'branch-edge-01'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "branch-edge-01", "site": "branch"}


### Example 0732 - Allowlist a public report: campus scenario

**Scenario 02/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'campus'` |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "campus-access-02", "site": "campus"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [32]:
# Example 0732: Allowlist a public report - campus
case = {'site': 'campus', 'hostname': 'campus-access-02'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "campus-access-02", "site": "campus"}


### Example 0733 - Allowlist a public report: dc scenario

**Scenario 03/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'dc'` |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "dc-leaf-03", "site": "dc"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [33]:
# Example 0733: Allowlist a public report - dc
case = {'site': 'dc', 'hostname': 'dc-leaf-03'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "dc-leaf-03", "site": "dc"}


### Example 0734 - Allowlist a public report: wan scenario

**Scenario 04/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'wan'` |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "wan-border-04", "site": "wan"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [34]:
# Example 0734: Allowlist a public report - wan
case = {'site': 'wan', 'hostname': 'wan-border-04'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "wan-border-04", "site": "wan"}


### Example 0735 - Allowlist a public report: lab scenario

**Scenario 05/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'lab'` |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "lab-router-05", "site": "lab"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [35]:
# Example 0735: Allowlist a public report - lab
case = {'site': 'lab', 'hostname': 'lab-router-05'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "lab-router-05", "site": "lab"}


### Example 0736 - Allowlist a public report: wireless scenario

**Scenario 06/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'wireless'` |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "wireless-dist-06", "site": "wireless"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [36]:
# Example 0736: Allowlist a public report - wireless
case = {'site': 'wireless', 'hostname': 'wireless-dist-06'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "wireless-dist-06", "site": "wireless"}


### Example 0737 - Allowlist a public report: remote scenario

**Scenario 07/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'remote'` |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "remote-gateway-07", "site": "remote"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [37]:
# Example 0737: Allowlist a public report - remote
case = {'site': 'remote', 'hostname': 'remote-gateway-07'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "remote-gateway-07", "site": "remote"}


### Example 0738 - Allowlist a public report: core scenario

**Scenario 08/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'core'` |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "core-router-08", "site": "core"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [38]:
# Example 0738: Allowlist a public report - core
case = {'site': 'core', 'hostname': 'core-router-08'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "core-router-08", "site": "core"}


### Example 0739 - Allowlist a public report: mgmt scenario

**Scenario 09/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'mgmt'` |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "mgmt-switch-09", "site": "mgmt"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [39]:
# Example 0739: Allowlist a public report - mgmt
case = {'site': 'mgmt', 'hostname': 'mgmt-switch-09'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "mgmt-switch-09", "site": "mgmt"}

### Example 0740 - Allowlist a public report: backup scenario

**Scenario 10/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'backup'` |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "backup-edge-10", "site": "backup"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [40]:
# Example 0740: Allowlist a public report - backup
case = {'site': 'backup', 'hostname': 'backup-edge-10'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "backup-edge-10", "site": "backup"}

## Pattern 075: Snapshot ownership at a boundary

**Core concept:** A method can return independent nested data rather than a live internal mapping.

**Network-engineering purpose:** A caller editing a report should not silently change stored interface intent.

### Read the implementation in this order
1. Store the model's nested state.
2. Deep-copy only serializable configuration data for a snapshot.
3. Modify the returned snapshot.
4. Check that internal state remains at its original value.

**Common mistake / boundary:** Do not deep-copy connection objects, locks, or other external resources.

**Practice after the ten worked scenarios:** Use frozen dataclasses and tuples to design an immutable snapshot instead.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0741 - Snapshot ownership at a boundary: branch scenario

**Scenario 01/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet1/1'` |
| `mtu` | `1500` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet1/1': {'mtu': 1500}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [41]:
# Example 0741: Snapshot ownership at a boundary - branch
case = {'interface': 'Ethernet1/1', 'mtu': 1500}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet1/1': {'mtu': 1500}}

### Example 0742 - Snapshot ownership at a boundary: campus scenario

**Scenario 02/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet2/1'` |
| `mtu` | `9000` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet2/1': {'mtu': 9000}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [42]:
# Example 0742: Snapshot ownership at a boundary - campus
case = {'interface': 'Ethernet2/1', 'mtu': 9000}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet2/1': {'mtu': 9000}}


### Example 0743 - Snapshot ownership at a boundary: dc scenario

**Scenario 03/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet3/1'` |
| `mtu` | `9216` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet3/1': {'mtu': 9216}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [43]:
# Example 0743: Snapshot ownership at a boundary - dc
case = {'interface': 'Ethernet3/1', 'mtu': 9216}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet3/1': {'mtu': 9216}}


### Example 0744 - Snapshot ownership at a boundary: wan scenario

**Scenario 04/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet4/1'` |
| `mtu` | `576` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet4/1': {'mtu': 576}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [44]:
# Example 0744: Snapshot ownership at a boundary - wan
case = {'interface': 'Ethernet4/1', 'mtu': 576}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet4/1': {'mtu': 576}}


### Example 0745 - Snapshot ownership at a boundary: lab scenario

**Scenario 05/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet5/1'` |
| `mtu` | `1600` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet5/1': {'mtu': 1600}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [45]:
# Example 0745: Snapshot ownership at a boundary - lab
case = {'interface': 'Ethernet5/1', 'mtu': 1600}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet5/1': {'mtu': 1600}}


### Example 0746 - Snapshot ownership at a boundary: wireless scenario

**Scenario 06/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet6/1'` |
| `mtu` | `1500` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet6/1': {'mtu': 1500}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [46]:
# Example 0746: Snapshot ownership at a boundary - wireless
case = {'interface': 'Ethernet6/1', 'mtu': 1500}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet6/1': {'mtu': 1500}}


### Example 0747 - Snapshot ownership at a boundary: remote scenario

**Scenario 07/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet7/1'` |
| `mtu` | `2000` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet7/1': {'mtu': 2000}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [47]:
# Example 0747: Snapshot ownership at a boundary - remote
case = {'interface': 'Ethernet7/1', 'mtu': 2000}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet7/1': {'mtu': 2000}}


### Example 0748 - Snapshot ownership at a boundary: core scenario

**Scenario 08/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet8/1'` |
| `mtu` | `4470` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet8/1': {'mtu': 4470}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [48]:
# Example 0748: Snapshot ownership at a boundary - core
case = {'interface': 'Ethernet8/1', 'mtu': 4470}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet8/1': {'mtu': 4470}}


### Example 0749 - Snapshot ownership at a boundary: mgmt scenario

**Scenario 09/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet9/1'` |
| `mtu` | `9100` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet9/1': {'mtu': 9100}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [49]:
# Example 0749: Snapshot ownership at a boundary - mgmt
case = {'interface': 'Ethernet9/1', 'mtu': 9100}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet9/1': {'mtu': 9100}}


### Example 0750 - Snapshot ownership at a boundary: backup scenario

**Scenario 10/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet10/1'` |
| `mtu` | `8192` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet10/1': {'mtu': 8192}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [50]:
# Example 0750: Snapshot ownership at a boundary - backup
case = {'interface': 'Ethernet10/1', 'mtu': 8192}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet10/1': {'mtu': 8192}}


## Check your understanding: Serialization, schemas, enums, and snapshots

**You should now be able to:** Export deliberate public fields, reject unsupported schema versions, and control snapshot ownership.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/tutorial/classes.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.